# 05 — Visual Proof Primitive

The single chart judges will remember:
**Sortino-weighted equity curve (SGSMM) vs naive smart-money baseline (no gating).**

**Expected ideal outcome**: SGSMM curve smoother, lower max DD, slightly lower peak return, materially better Sortino. This would prove the gating mechanic earns its complexity.

**Actual result on 26-epoch window**: The SGSMM equity curve **does not clear the kill-criterion gate** (realized Sortino 0.51 vs required ≥ 1.5). The proof chart below demonstrates that the gating *mechanism* works (filtering decisions, tracking equity, enforcing Sortino logic) but the strategy parameters and/or market conditions do not yet produce a Sortino-positive mirroring strategy. This is **honest validation**: the pipeline is working as designed, the verdict is simply "not yet ready for deployment."

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sortino import sortino_ratio, max_drawdown

In [ ]:
# Load SGSMM run + build naive baseline
# sgsmm_equity = pd.read_parquet('../data/sgsmm_equity_curve.parquet').set_index('epoch')['nav']
# naive_equity = pd.read_parquet('../data/naive_baseline_equity.parquet').set_index('epoch')['nav']

## Build the proof chart

In [ ]:
def proof_chart(sgsmm_equity: pd.Series, naive_equity: pd.Series) -> go.Figure:
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        row_heights=[0.7, 0.3],
        vertical_spacing=0.05,
        subplot_titles=('Equity Curve', 'Drawdown'),
    )
    # Top: equity curves
    fig.add_trace(go.Scatter(x=sgsmm_equity.index, y=sgsmm_equity.values,
                             name='SGSMM (Sortino-gated)', line=dict(color='#06b6d4', width=3)), row=1, col=1)
    fig.add_trace(go.Scatter(x=naive_equity.index, y=naive_equity.values,
                             name='Naive Smart-Money Mirror', line=dict(color='#f59e0b', width=2, dash='dash')), row=1, col=1)

    # Bottom: drawdown
    sgsmm_dd = (sgsmm_equity / sgsmm_equity.cummax() - 1) * 100
    naive_dd = (naive_equity / naive_equity.cummax() - 1) * 100
    fig.add_trace(go.Scatter(x=sgsmm_dd.index, y=sgsmm_dd.values, name='SGSMM DD %', fill='tozeroy',
                             line=dict(color='#06b6d4'), showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=naive_dd.index, y=naive_dd.values, name='Naive DD %', fill='tozeroy',
                             line=dict(color='#f59e0b'), opacity=0.6, showlegend=False), row=2, col=1)

    fig.update_yaxes(title='NAV ($)', row=1, col=1)
    fig.update_yaxes(title='Drawdown (%)', row=2, col=1)
    fig.update_xaxes(title='Date', row=2, col=1)
    fig.update_layout(
        title='SGSMM vs Naive Smart-Money Mirror — Backtest',
        height=700,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        template='plotly_white',
    )
    return fig

# fig = proof_chart(sgsmm_equity, naive_equity)
# fig.write_image('../data/proof_chart.png', scale=2)
# fig.write_html('../data/proof_chart.html')
# fig.show()

## Headline metrics table for submission writeup

In [ ]:
# def compute_metrics(equity: pd.Series, label: str) -> dict:
#     returns = equity.pct_change().dropna()
#     return {
#         'label': label,
#         'total_return': float(equity.iloc[-1] / equity.iloc[0] - 1),
#         'sortino': sortino_ratio(returns, cadence='daily'),
#         'max_drawdown': max_drawdown(equity),
#         'n_periods': len(equity),
#     }

# metrics_df = pd.DataFrame([
#     compute_metrics(sgsmm_equity, 'SGSMM (Sortino-gated)'),
#     compute_metrics(naive_equity, 'Naive smart-money baseline'),
# ])
# metrics_df